# Course 1: Geometric Deep Learning and Graphs
## Session 10: Graphs meet LLMs

**Goal:** compare plain retrieval with graph retrieval on a small medical knowledge graph, using multi-hop questions.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. A small knowledge graph
Triples of (head, relation, tail). The facts are simplified for teaching and are not medical advice. Each triple also becomes one sentence, which is the text store for plain RAG.

In [ ]:
import numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer

triples = [
 ("dengue", "is spread by", "aedes mosquito"), ("chikungunya", "is spread by", "aedes mosquito"),
 ("malaria", "is spread by", "anopheles mosquito"), ("aedes mosquito", "breeds in", "standing clean water"),
 ("anopheles mosquito", "breeds in", "marshes and rice fields"), ("dengue", "has symptom", "high fever"),
 ("dengue", "has symptom", "low platelet count"), ("malaria", "has symptom", "cyclic fever"),
 ("cholera", "is spread by", "contaminated water"), ("cholera", "has symptom", "watery diarrhoea"),
 ("cholera", "is treated with", "oral rehydration salts"), ("oral rehydration salts", "contain", "glucose and sodium"),
 ("typhoid", "is spread by", "contaminated food"), ("typhoid", "is treated with", "azithromycin"),
 ("azithromycin", "belongs to", "macrolide antibiotics"), ("tuberculosis", "is treated with", "rifampicin"),
 ("rifampicin", "can harm", "the liver"), ("tuberculosis", "has symptom", "long cough"),
 ("malaria", "is treated with", "artemisinin combination therapy"), ("artemisinin combination therapy", "comes from", "sweet wormwood"),
 ("type 2 diabetes", "is treated with", "metformin"), ("metformin", "can cause", "stomach upset"),
 ("type 2 diabetes", "raises the risk of", "kidney disease"), ("hypertension", "raises the risk of", "stroke"),
 ("hypertension", "is treated with", "amlodipine"), ("amlodipine", "can cause", "ankle swelling"),
]
sentences = [f"{h} {r} {t}." for h, r, t in triples]

# Multi-hop questions: the answer is two steps from the entity named in the question.
questions = [
 ("Where does the insect that spreads dengue breed?", "standing clean water"),
 ("Where does the insect that spreads malaria breed?", "marshes and rice fields"),
 ("What do the salts used for cholera contain?", "glucose and sodium"),
 ("Which drug family does the typhoid medicine belong to?", "macrolide antibiotics"),
 ("Which organ can the tuberculosis medicine harm?", "the liver"),
 ("What plant does the malaria therapy come from?", "sweet wormwood"),
 ("What side effect can the type 2 diabetes medicine cause?", "stomach upset"),
 ("What side effect can the hypertension medicine cause?", "ankle swelling"),
]
print(len(triples), "triples,", len(questions), "questions")

### 2. Plain RAG: retrieve the top-k most similar sentences

In [ ]:
vec = TfidfVectorizer().fit(sentences); S = vec.transform(sentences)

def plain_rag(q, k=2):
    scores = (S @ vec.transform([q]).T).toarray().ravel()
    return [sentences[i] for i in np.argsort(-scores)[:k]]

print(questions[0][0]); print(" ->", plain_rag(questions[0][0]))

### 3. Graph RAG: find the entity in the question, then walk the graph
This is the local-search idea: start from entities mentioned in the question and collect facts within two hops. To keep the comparison fair, the graph retriever is also limited to k sentences.

In [ ]:
out_edges = defaultdict(list)
for i, (h, r, t) in enumerate(triples): out_edges[h].append((i, t))
entities = sorted({h for h, _, _ in triples} | {t for _, _, t in triples}, key=len, reverse=True)

def graph_rag(q, k=2, hops=2):
    seeds = [e for e in entities if e in q.lower()]
    frontier, found = set(seeds), []
    for _ in range(hops):
        nxt = set()
        for e in frontier:
            for i, t in out_edges[e]: found.append(i); nxt.add(t)
        frontier = nxt
    if not found: return plain_rag(q, k)
    qv = vec.transform([q]); sim = lambda i: (S[i] @ qv.T).toarray()[0, 0]
    # every two-step path that starts at an entity named in the question, scored by how well both facts match the question
    paths = [(sim(i) + sim(j), i, j) for e in seeds for i, t in out_edges[e] for j, _ in out_edges[t]]
    if not paths: return plain_rag(q, k)
    paths.sort(reverse=True)
    picked = []
    for _, i, j in paths:
        for f in (j, i):                                      # the second fact holds the answer, so keep it first
            if f not in picked: picked.append(f)
    return [sentences[i] for i in picked[:k]]

print(questions[0][0]); print(" ->", graph_rag(questions[0][0]))

### 4. Score both: does the retrieved context contain the answer?
No LLM is used here. If the answer is not in the context, a model that sticks to the context cannot produce it.

In [ ]:
def recall(retriever, k):
    return np.mean([any(ans in s for s in retriever(q, k)) for q, ans in questions])

for k in [1, 2, 3]:
    print(f"k = {k}:  plain RAG {recall(plain_rag, k):.2f}   graph RAG {recall(graph_rag, k):.2f}")
assert recall(graph_rag, 2) > recall(plain_rag, 2)

Why plain retrieval struggles: the sentence holding the answer (for example 'aedes mosquito breeds in standing clean water') shares almost no words with the question, which mentions dengue. The graph supplies the missing link.

### 5. Add the generator (TODO, NOT RUN here)

In [ ]:
# NOT RUN: needs a model. Any small open instruction model on Kaggle or Colab will do.
def answer(question, context, generate):
    prompt = "Answer using only the facts below. If the facts are not enough, say 'not enough information'.\n\n"
    prompt += "\n".join("- " + c for c in context) + f"\n\nQuestion: {question}\nAnswer:"
    return generate(prompt)
# for q, gold in questions: compare answer(q, plain_rag(q), generate) with answer(q, graph_rag(q), generate)
# Report exact-match accuracy for both, and count how often each one says 'not enough information'.

### TODO
1. Write 8 single-hop questions. Does graph RAG still help, or is plain RAG enough?
2. Add 20 distractor triples that reuse the same words. Which method degrades faster?
3. The GraphRAG paper also builds community summaries for 'global' questions. Write two global questions about this graph that neither retriever here can answer.